# MRDRP Dashboard Launcher — Backend MR for any analysis set (P2)

Current stage only: deploys the two pending pieces for the new "Run MR for a saved analysis set" feature (`mr_pipeline.py` + the Backend MR Results page patch), then launches the dashboard. It assumes `app.py`, `gwas_catalog_client.py`, and `gwas_catalog_ftp.py` are already in place from earlier stages -- this notebook does not touch the GWAS Catalog Search page or re-apply any of that history.

**Important -- two separate notebooks, two separate jobs:**
- **This notebook** only needs plain Python + Streamlit. It does **not** need R, rpy2, TwoSampleMR, or an OpenGWAS token -- the dashboard only *reads* result files, it never runs the MR computation itself.
- **`Python_based_MR_drug_repurposing_pipeline.ipynb`** (your separate, existing notebook) is where the actual R/TwoSampleMR computation runs, using `mr_pipeline.run_pipeline_for_analysis_set(...)` -- that notebook still needs all of its existing R/rpy2/OpenGWAS setup cells.

Cells below are kept small/granular on purpose so a slow step doesn't stall the whole run.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

project_path = Path("/content/drive/MyDrive/UM_WQF7023/MRDRP-main")
app_path = project_path / "app.py"

print("Project path:", project_path)
print("app.py exists:", app_path.exists())

Project path: /content/drive/MyDrive/UM_WQF7023/MRDRP-main
app.py exists: True


## 2. Sanity-check earlier stages are already in place

This notebook builds on top of the GWAS Catalog Search stage. If any of these are missing, go back and run that stage's deployment cells first -- this notebook does not recreate them.

In [ ]:
required_files = ["app.py", "gwas_catalog_client.py", "gwas_catalog_ftp.py", "analysis_set_record.csv"]
missing = [f for f in required_files if not (project_path / f).exists()]

if missing:
    raise FileNotFoundError(
        f"Missing from {project_path}: {', '.join(missing)}. "
        "Please complete the earlier GWAS Catalog Search / Analysis Set Selection stages first."
    )

print("All required files from earlier stages are present.")

All required files from earlier stages are present.


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/add_delete_files_and_download.py"

Backup created:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app_backup_before_delete_and_download_20260806_101328.py
Applied: Add _cached_read_file_bytes helper
Applied: Add download button for the full selected file
Applied: Targeted File Screening: delete/clear now also delete underlying files
Applied: Analysis Set Selection: delete/clear now also delete results folders
All 4 edits applied: delete buttons on both pages now also permanently delete
the underlying file(s)/folder(s), not just the record row; Targeted File
Screening also has a new full-file download button.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/write_mr_pipeline_to_drive.py"

mr_pipeline.py updated -- CRITICAL PERFORMANCE FIX: the last-resort outcome
fallback (Tier 3) no longer attempts Ensembl rsID mapping across an entire
outcome file. That was doing one network lookup PER ROW with an enforced delay
between each -- for a real, unfiltered outcome file with hundreds of thousands
of rows, this could take hours. It now fails fast instead, with a clear message,
when a file lacks native rsIDs -- exactly what Tier 1/Tier 2 position-matching
already handle correctly and efficiently.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/mr_pipeline.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/write_gwas_catalog_ftp_to_drive.py"

gwas_catalog_ftp.py updated -- filter_significant_from_local_file() can now
derive p-value on the fly (from beta/odds_ratio + standard_error) when a study
has no native p_value column, instead of failing outright. This closes the gap
where mr_pipeline.py could already handle a missing p-value for an ALREADY-SAVED
file, but saving as exposure via the GWAS Catalog Search page had no equivalent.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/gwas_catalog_ftp.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/write_mr_pipeline_to_drive.py"

mr_pipeline.py updated -- exposure-side rsID mapping (ensure_rsid_column) now
tries BATCHED Ensembl lookups first (up to 150 positions per HTTP request via the
VEP endpoint) instead of one request per row, falling back to the original
per-row lookup only for whatever a batch fails on. Also added progress printing
throughout so a genuinely slow run is visibly progressing, not silent.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/mr_pipeline.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/drop_unused_columns.py"

Backup created:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/analysis_set_record_backup_before_column_cleanup_20260809_094556.csv

Current columns (14): ['analysis_set_name', 'exposure_files', 'exposure_traits', 'required_exposures', 'missing_exposure_traits', 'outcome_file', 'outcome_trait', 'build_route', 'build_match', 'set_status', 'notes', 'exposure_trait_counts', 'outcome_files', 'outcome_traits']

Dropped 4 confirmed-empty column(s): ['required_exposures', 'missing_exposure_traits', 'outcome_file', 'outcome_trait']
Remaining columns (10): ['analysis_set_name', 'exposure_files', 'exposure_traits', 'build_route', 'build_match', 'set_status', 'notes', 'exposure_trait_counts', 'outcome_files', 'outcome_traits']


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/add_snp_count_filter.py"

Backup created:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app_backup_before_snp_count_filter_20260809_094617.py
Patch applied: the study list table now has an opt-in checkbox to filter out
studies below 1,000,000 SNPs -- helps exclude exome-restricted/narrow-panel
studies (like the UK Biobank T2D exome batch) that would otherwise silently
waste a full pipeline run before failing with zero outcome matches.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/patch_8_cpi_gnn_p2_dashboard.py"

Backup created:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app_backup_before_cpi_p2_dashboard_20260809_101928.py
Patch applied: CPI / GNN Exploration page extended with P2 sections
(3D conformer info, real protein sequences, new 'Artesunate-target
prediction (P2)' tab).
Reload the Streamlit app (rerun the launch cell / refresh the tunnel URL) to see it.


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/write_gwas_catalog_client_to_drive.py"

gwas_catalog_client.py updated -- added the missing GwasCatalogRateLimitError class
(with a .retry_after attribute) that app.py already expected but was never defined.
_get_json() now raises this specific error on a 429 response instead of the generic
GwasCatalogAPIError, fixing the AttributeError that was masking the real rate-limit
message on the GWAS Catalog Search page.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/gwas_catalog_client.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/write_mr_pipeline_to_drive.py"

mr_pipeline.py updated on Drive/Colab -- two changes:
1) Reverted to per-row Ensembl lookup (the batch endpoint was silently returning
   a 0% match rate -- confirmed 2026-08-13 on a real server run).
2) Added caching: a clumped exposure result is now reused on subsequent runs of
   the same analysis set, as long as the source file is unchanged and the
   p_threshold matches -- pass use_cache=False to force a full recompute.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/mr_pipeline.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/add_api_error_fallback.py"

Backup created:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app_backup_before_api_error_fallback_20260813_104243.py
Applied: Broaden the 'check summary stats' except block to catch generic API errors too
Applied: Broaden the 'Preview significant variants' except block the same way
All edits applied: a non-rate-limit GWAS Catalog API error (e.g. a 500 from EBI's
server, as just happened) no longer crashes the whole GWAS Catalog Search page --
it now falls back to the existing FTP route gracefully, same as when the REST API
simply doesn't have a study loaded.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/app.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/write_gwas_catalog_client_to_drive.py"

gwas_catalog_client.py updated -- an HTML error page from a non-2xx response
(e.g. a real 500 from EBIs own server, as just happened for GCST90043892) no
longer dumps raw markup into the error message. The failure is still fully shown
(status code, URL, plain-English reason) -- only the useless HTML noise is
dropped. A genuinely useful error body (JSON/plain text from a well-behaved API)
is still shown in full, unchanged.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/gwas_catalog_client.py


In [ ]:
!python3 "/content/drive/MyDrive/UM_WQF7023/MRDRP-main/add_download_progress_callback.py"

Backup created:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/gwas_catalog_ftp_backup_before_progress_bar_20260821_141133.py
Applied: Add progress_callback support to download_file()
Applied: Add progress_callback parameter to fetch_significant_from_ftp()'s signature
Applied: Pass progress_callback through to download_file() at the call site
All edits applied: download_file() and fetch_significant_from_ftp() now accept an
optional progress_callback(downloaded_bytes, total_bytes_or_None), letting the
dashboard drive a real progress bar during large FTP downloads instead of a plain spinner.
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/gwas_catalog_ftp.py


## 5. Install the dashboard's Python dependencies

Just Streamlit + pyliftover (used by the FTP-download page's automatic liftover step) -- no R needed here. Both need reinstalling every fresh Colab runtime, since nothing but Drive persists across sessions.

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.8 MB/s eta 0:00:00


In [ ]:
!pip install -q pyliftover --break-system-packages

## 6. Launch the dashboard

In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true

^C
^C


In [ ]:
!streamlit run /content/drive/MyDrive/UM_WQF7023/MRDRP-main/app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(5)

!cat /content/streamlit.log



2026-08-22 04:04:26.355 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.19.52.131:8501



In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Sat, 22 Aug 2026 04:04:36 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 11141
last-modified: Sat, 22 Aug 2026 04:03:57 GMT
etag: "2bf0d62fe17cb3e60d0ebbc0609e3973"
cache-control: no-cache



## 7. Expose the dashboard via a Cloudflare Tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
!nohup /content/cloudflared tunnel --url http://localhost:8501 > /content/streamlit_tunnel.log 2>&1 &

In [ ]:
import time
import re

time.sleep(8)

with open("/content/streamlit_tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
    log_text = f.read()

print(log_text)

urls = re.findall(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_text)

if len(urls) == 0:
    raise RuntimeError("No trycloudflare URL found. Please wait a few seconds and run this cell again.")

streamlit_url = urls[0]
print("Streamlit public URL:")
print(streamlit_url)

2026-08-22T04:04:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-22T04:04:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-22T04:04:51Z INF +--------------------------------------------------------------------------------------------+
2026-08-22T04:04:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-22T04:04:51Z INF |  https://glucose-spice-enabled-forth.trycloudflare.com

## 8. Test this stage

Open the URL printed above, click **"Backend MR Results"** in the sidebar, and look for the new **"0) Run MR for a saved analysis set (P2)"** section at the top. Pick one of your saved analysis sets from the dropdown -- it will say results don't exist yet (unless you've already run the pipeline notebook for it) and show you the exact `mr_pipeline.run_pipeline_for_analysis_set(...)` call to paste into `Python_based_MR_drug_repurposing_pipeline.ipynb`.

Once you've run that in the pipeline notebook, come back and refresh this page -- the results, clumping summary, and run summary should appear.

If anything above raised an error, or the new section doesn't show up, paste the traceback / a screenshot back.